# Réseau de neurones convolutif (CNN)

## 1. Imports necéssaires pour ce projet

In [1]:
import pickle
import numpy as np
import copy
import os
import logging
from sklearn.preprocessing import LabelEncoder
from datetime import datetime
from sklearn import metrics

## 2. Paramètres de configuration

In [2]:
RATIO_DATASET: float = 1.0
"""Pourcentage du dataset qui est utilisé"""
WIDTH: int = 40
"""Largeur en pixels des imgs à l'entrée de la première couche de convolution"""
HEIGHT: int = 40
"""Hauteur en pixels des imgs à l'entrée de la première couche de convolution"""
EPOCHS: int = 15
"""Nombre d'itérations sur les données d'entraînement pour améliorer le modèle"""
BATCHES_SIZE: int = 32
"""Nombre de données sur lesquelles le modèle s'entraine par groupe"""
PATIENCE: int = 3
"""Nombre d'epochs à attendre avant d'arrêter l'entraînement si aucune amélioration n'est constatée"""
LABELS_Y = [
    "Cercle3",
    "Cercle4",
    "Diamant3",
    "Diamant4",
    "Hexagone3",
    "Hexagone4",
    "Triangle3",
    "Triangle4",
]

## 3. Chargement du dataset

Récuperation des données d'entrainement sérialisées grâce à la librairie pickle

In [3]:
X_dataset_imgs = None
"""imgs d'entrée de dimensions WIDTH * HEIGHT"""
y_dataset_labels = None
"""labels liés à chaque img d'entrée"""
filenames_serialized = ""

with open("X.pickle", "rb") as f:
    X_serialized = pickle.load(f)
with open("y.pickle", "rb") as f:
    y_serialized = pickle.load(f)
with open("filenames.pickle", "rb") as f:
    filenames_serialized = pickle.load(f)

label_encoder = LabelEncoder()
y_training_encoded = label_encoder.fit_transform(y_serialized)

print(
    "# image, image sur 1 vecteur:",
    X_serialized.shape,
)
print("images et etiquettes respectives:", len(y_training_encoded))
print("images et etiquettes respectives:", len(filenames_serialized))

# image, image sur 1 vecteur: (11632, 1600)
images et etiquettes respectives: 11632
images et etiquettes respectives: 11632


In [4]:
NB_OF_IMGS = int(len(X_serialized) * RATIO_DATASET)
"""nombre d'imgs pour faire l'entrainement"""

# convertir les vecteurs en type numpy
X_dataset_imgs = X_serialized[:NB_OF_IMGS]
X_dataset_imgs = np.array(X_dataset_imgs, dtype=np.float32)
print("images d'entrée:", X_dataset_imgs.shape)

y_dataset_labels = y_training_encoded[:NB_OF_IMGS]
y_dataset_labels = np.array(y_dataset_labels)
print("étiquettes d'entrée:", y_dataset_labels.shape)

X_dataset_imgs = X_dataset_imgs.reshape(-1, HEIGHT, WIDTH)

images d'entrée: (11632, 1600)
étiquettes d'entrée: (11632,)


## 4. Normalisation des données

Ici, les valeurs comprises entre 0 et 255 sont normalisées par la division de 255.0

In [5]:
X_dataset_imgs *= 1 / 255.0

## 5. Division (Hold-Out) entrainement (60%)/ validation (20%)/ test (20%)

Les ratio de 60% pour l'entrainement, 20% pour la validation et 20% pour les tests sont gardés dans ces variables

In [6]:
PCT_60 = int(0.6 * len(X_dataset_imgs))
PCT_20 = int(0.2 * len(X_dataset_imgs))

Les ndarrays précédents (X_dataset_imgs, y_dataset_labels) sont séparés selon les ratios du Hold-Out

In [7]:
# list slicing du contenu du dataset en pourcentages
X_TRAINING = X_dataset_imgs[:PCT_60]
Y_TRAINING = y_dataset_labels[:PCT_60]

X_VALIDATION = X_dataset_imgs[PCT_60 : PCT_60 + PCT_20]
Y_VALIDATION = y_dataset_labels[PCT_60 : PCT_60 + PCT_20]

X_TEST = X_dataset_imgs[PCT_60 + PCT_20 :]
Y_TEST = y_dataset_labels[PCT_60 + PCT_20 :]

## 6. Fonctions utilitaires

In [8]:
# Implémentation du mécanisme d'extraction de kernels d'une image
def extract_region(img, h, w, size):
    return img[:, h : h + size, w : w + size, :]

In [9]:
def patch_iteration(img, size):
    limit_h = img.shape[1] - size + 1
    limit_w = img.shape[2] - size + 1

    # Itère chaque pixel excluant les derniers pixels en fonction de size pour éviter débordement
    for h in range(limit_h):
        for w in range(limit_w):
            # Slice l'image de h à h+size et de w à w+size pour obtenir un patch de size x size
            yield extract_region(img, h, w, size), h, w

## 7. Implémentation de la couche de convolution

In [10]:
class Convolution:
    def __init__(self, num_filters, size):
        self.num_filters = num_filters
        self.size = size
        # --------------------
        self.input_img = None
        self.weights = None

    def init_weights(self, nb_rgb_ch):
        random_nd_array = np.random.randn(
            self.num_filters, self.size, self.size, nb_rgb_ch
        )
        self.weights = random_nd_array / (self.size * self.size)

    def forward(self, img):
        # Si 2D, ajouter une nouvelle dimension pour les channels (ex: (H, W) -> (H, W, 1))
        if img.ndim == 3:
            img = img[:, :, :, np.newaxis]
        self.input_img = img
        nb, img_h, img_w, channels = img.shape

        # Initialise les poids selon les nombres de channels, 1 pour premier layer
        if self.weights is None:
            self.init_weights(channels)

        output = np.zeros(
            (nb, img_h - self.size + 1, img_w - self.size + 1, self.num_filters)
        )

        # Fait un dot entre le patch de l'image d'entrée et les kernels pour chaque position (h, w) et chaque kernel
        for patch, h, w in patch_iteration(img, self.size):  # acceptable
            output[:, h, w, :] = np.tensordot(
                self.weights, patch, axes=([1, 2, 3], [1, 2, 3])
            ).T

        return output

    def back(self, grad_output, learning_rate):
        grad_weights = np.zeros_like(self.weights)
        grad_input = np.zeros_like(self.input_img)

        for patch, h, w in patch_iteration(self.input_img, self.size):
            # Gradient pour chaque filtre
            grad_val = grad_output[:, h, w, :]

            # Ajuste les poids des kernels avec le gradient du patch de l'image d'entrée
            grad_weights += np.tensordot(grad_val, patch, axes=([0], [0]))

            # Ajuste le gradient de l'image d'entrée à la position (h, w) avec un dot entre le gradient du filtre et les poids
            grad_input[:, h : h + self.size, w : w + self.size, :] += np.tensordot(
                grad_val, self.weights, axes=([1], [0])
            )

        self.weights -= learning_rate * grad_weights
        return grad_input

## 8. Implémentation du Maxpooling

In [11]:
class MaxPooling:
    def __init__(self, pool_size):
        self.size = pool_size
        self.input = None

    def forward(self, img):
        # Si 2D, ajouter une nouvelle dimension pour les channels (ex: (H, W) -> (H, W, 1))
        if img.ndim == 3:
            img = img[:, :, :, np.newaxis]
        self.input = img
        nb, img_h, img_w, channels = img.shape
        out_h = img_h // self.size
        out_w = img_w // self.size

        # Trim l'image pour que ses dimensions soient divisibles par kernel_size
        img_trimmed = img[:, : out_h * self.size, : out_w * self.size, :]
        # Reshape image en patchs de pooling
        img_trimmed = img_trimmed.reshape(
            nb, out_h, self.size, out_w, self.size, channels
        )
        # Maximise selon les dimensions du kernel
        max_pooling_output = img_trimmed.max(axis=(2, 4))

        return max_pooling_output

    def back(self, grad_output, learning_rate=None):
        nb, img_h, img_w, channels = self.input.shape
        out_h = img_h // self.size
        out_w = img_w // self.size

        # Comme dans forward_prop
        img_trimmed = self.input[:, : out_h * self.size, : out_w * self.size, :]
        img_trimmed = img_trimmed.reshape(
            nb, out_h, self.size, out_w, self.size, channels
        )
        max_pooling_output = img_trimmed.max(axis=(2, 4), keepdims=True)

        # Crée un masque pour les positions des max dans chaque patch
        mask = img_trimmed == max_pooling_output

        # Reshape grad_output pour correspondre à la forme du patch de pooling
        grad_output_expanded = grad_output[:, :, np.newaxis, :, np.newaxis, :]

        # Applique le gradient seulement aux max positions
        grad_input_expanded = mask * grad_output_expanded

        # Reshape à la forme originale de l'image d'entrée
        grad_input = grad_input_expanded.reshape(
            nb, out_h * self.size, out_w * self.size, channels
        )

        # Pad pour les dimensions qui ont été trimées
        grad_input_full = np.zeros_like(self.input)
        grad_input_full[:, : out_h * self.size, : out_w * self.size, :] = grad_input

        return grad_input_full

## 9. Implémentation de la fonction d'activation ReLU

In [12]:
class ReLU:
    def __init__(self):
        self.img = None

    def forward(self, img):
        self.img = img
        # les parties de l'image qui sont négatives sont mises à 0 et les parties positives restent identiques
        return np.maximum(0, img)

    def back(self, gradient):
        # le gradient passe seulement pour les parties qui sont positives
        grad_input = gradient * (self.img > 0)
        return grad_input

## 10. Implémentation de la couche de sortie avec Softmax

In [13]:
class Softmax:
    def __init__(self, input_size, output_size):
        self.weight = np.random.randn(input_size, output_size) / input_size
        self.bias = np.zeros(output_size)
        self.output = None
        self.input_flat = None
        self.original_input = None

    def forward(self, img: np.ndarray):
        self.original_input = img.shape

        nb, img_h, img_w, channels = img.shape
        img_flat = img.reshape(nb, img_h * img_w * channels)
        self.input_flat = img_flat

        output = np.dot(img_flat, self.weight) + self.bias
        self.output = output

        # Empeche une explosion causée par l'exponentielle
        shifted = output - np.max(output, axis=1, keepdims=True)  # watch out
        exp_vals = np.exp(shifted)

        # Activation
        softmax_output = exp_vals / np.sum(exp_vals, axis=1, keepdims=True)
        return softmax_output

    def back(self, grad_loss_output, learning_rate):
        s = np.exp(self.output - np.max(self.output, axis=1, keepdims=True))
        softmax_out = s / np.sum(s, axis=1, keepdims=True)

        grad_input_softmax = softmax_out * (
            grad_loss_output
            - (grad_loss_output * softmax_out).sum(axis=1, keepdims=True)
        )

        grad_weights = self.input_flat.T @ grad_input_softmax  # (input_dim, output_dim)
        grad_bias = np.sum(grad_input_softmax, axis=0)  # (output_dim,)

        grad_input = grad_input_softmax @ self.weight.T

        # Update parameters
        self.weight -= learning_rate * grad_weights
        self.bias -= learning_rate * grad_bias

        return grad_input.reshape(self.original_input)

## 11. Définition de l'architecture du réseau de neurones

In [14]:
LAYERS = [
    Convolution(32, 3),
    ReLU(),
    MaxPooling(2),
    Convolution(64, 3),
    ReLU(),
    MaxPooling(2),
    Softmax(
        4096, len(LABELS_Y)
    ),  # softmax layer with flattened_size input and 16 output
]

## 12. Propagation avant

In [15]:
def forward_pass(x_input_nd_array, y_nd_array, layers):
    for layer in layers:
        x_input_nd_array = layer.forward(x_input_nd_array)
    # Compute loss (cross-entropy) and accuracy
    loss = -np.mean(
        np.log(
            x_input_nd_array[np.arange(x_input_nd_array.shape[0]), y_nd_array] + 1e-15
        )
    )
    predicition = np.argmax(x_input_nd_array, axis=1)
    return x_input_nd_array, loss, predicition

## 13. Rétropropagation

In [16]:
def backward_pass(initial_gradient, layers, learning_rate):
    grad = initial_gradient
    for layer in reversed(layers):
        if isinstance(layer, (Convolution, Softmax)):
            grad = layer.back(grad, learning_rate)
        elif isinstance(layer, (MaxPooling, ReLU)):
            grad = layer.back(grad)
    return grad

## 14. Fonction d'entraînement

In [17]:
def train(x_ndarray, y_ndarray, layers, learning_rate):
    # propagation avant
    output, loss, prediction = forward_pass(x_ndarray, y_ndarray, layers)

    nb, num_classes = output.shape
    gradient = output.copy()  # contient le output de softmax
    gradient[np.arange(nb), y_ndarray] -= 1  # grad = softmax - one_hot
    gradient /= nb

    # Rétropropagation
    backward_pass(gradient, layers, learning_rate)

    return output, loss, prediction

In [18]:
def train_epoch_batches(X, Y, layers, learning_rate):
    all_predictions = []
    all_losses = []

    for i in range(0, len(X), BATCHES_SIZE):
        x_batch = X[i : i + BATCHES_SIZE]
        y_batch = Y[i : i + BATCHES_SIZE]
        output, loss, prediction = train(x_batch, y_batch, layers, learning_rate)

        all_predictions.extend(prediction)
        all_losses.append(np.mean(loss))

    # Convert lists to numpy arrays
    all_predictions = np.array(all_predictions)
    all_losses = np.array(all_losses)

    accuracy = metrics.accuracy_score(Y, all_predictions)
    report = metrics.classification_report(
        Y, all_predictions, target_names=LABELS_Y, zero_division=0
    )
    mean_loss = np.mean(all_losses)

    return mean_loss, accuracy, report

## 15. Évaluation du modèle


In [19]:
def evaluate(X, Y, layers):
    output, loss, prediction = forward_pass(X, Y, layers)
    accuracy = metrics.accuracy_score(Y, prediction)
    conf_mat = metrics.confusion_matrix(Y, prediction)
    report = metrics.classification_report(
        Y, prediction, target_names=LABELS_Y, zero_division=0
    )
    loss = np.mean(loss)

    return loss, accuracy, conf_mat, report

## 16. Implémentation de la télémétrie

In [20]:
folder_path = f"best_models/{len(LAYERS) + 1}_{RATIO_DATASET * 100}"
os.makedirs(folder_path, exist_ok=True)

In [21]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(message)s",
    handlers=[
        logging.FileHandler(f"{folder_path}/training.log", mode="w"),
        logging.StreamHandler(),
    ],
    force=True,
)
log = logging.info

## 17. Entraînement du modèle


In [22]:
epochs_without_improvement = 0
best_layers = None
layer_filename = None

In [ ]:
start = datetime.now()
best_accuracy = 0
for epoch in range(EPOCHS):
    ln_rate = 0.15 / (1 + 0.05 * epoch)
    # ------------------------------------------------------------------------
    log(f"Epoch {epoch + 1}")
    loss_train, accuracy_train, report_train = train_epoch_batches(
        X_TRAINING, Y_TRAINING, LAYERS, ln_rate
    )
    log(f"learning rate {ln_rate}")
    log(f"Training: Loss moyen: {loss_train}, Accuracy moyen: {accuracy_train}")
    log(f"Classification Report:\n{report_train}")
    _, accuracy_val, _, _ = evaluate(X_VALIDATION, Y_VALIDATION, LAYERS)
    # ------------------------------------------------------------------------
    if accuracy_val > best_accuracy:
        log(f"Modèle validé avec une accuracy de {accuracy_val} vs {best_accuracy}")
        epochs_without_improvement = 0
        best_accuracy = accuracy_val
        best_layers = copy.deepcopy(LAYERS)

        layer_filename = f"model_{NB_OF_IMGS}_epoch_{epoch + 1}.pickle"
        with open(f"{folder_path}/{layer_filename}", "wb") as f:
            pickle.dump(best_layers, f)
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            log(f"Arrêt de l'entraînement après {PATIENCE} epochs sans amélioration")
            break
        log(f"Modèle non amélioré : {accuracy_val} vs meilleur {best_accuracy}")
end = datetime.now()
log(f"Temps total d'entraînement: {end - start}")

2026-03-20 00:04:22,342 - Epoch 1
2026-03-20 00:04:49,722 - learning rate 0.15
2026-03-20 00:04:49,723 - Training: Loss moyen: 1.6813518718451215, Accuracy moyen: 0.39403926063906003
2026-03-20 00:04:49,724 - Classification Report:
              precision    recall  f1-score   support

     Cercle3       0.43      0.41      0.42       870
     Cercle4       0.50      0.77      0.60       836
    Diamant3       0.37      0.23      0.29       877
    Diamant4       0.31      0.28      0.30       872
   Hexagone3       0.40      0.20      0.27       876
   Hexagone4       0.43      0.67      0.53       872
   Triangle3       0.30      0.24      0.26       836
   Triangle4       0.33      0.36      0.34       940

    accuracy                           0.39      6979
   macro avg       0.38      0.40      0.38      6979
weighted avg       0.38      0.39      0.37      6979

2026-03-20 00:04:53,641 - Modèle validé avec une accuracy de 0.5266552020636286 vs 0
2026-03-20 00:04:56,529 - Epoch 

## 18. Évaluation finale du meilleur modèle

In [ ]:
loss_test, accuracy_test, confusion_matrix, report_test = evaluate(
    X_TEST, Y_TEST, best_layers
)
log(
    f"Test terminée pour {layer_filename}: Accuracy: {accuracy_test}\n Confusion Matrix:\n{confusion_matrix}"
)
log(f"Classification Report:\n{report_test}")